In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "Data"
RAW = ROOT / "PAMAP2_Dataset" / "Protocol"

EXPECTED = {
    "training_rows": 229015,
    "reporting_rows": 58242,
    "training_participants": [
        "subject101", "subject102", "subject105",
        "subject106", "subject107", "subject108", "subject109"
    ],
    "reporting_participants": ["subject103", "subject104"]
}

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Working directory:", ROOT)
print("Official Data/ exists:", DATA.is_dir())

Python: 3.14.7
pandas: 3.0.6
Working directory: /Users/maxmagrill/DS4021-Final-Project
Official Data/ exists: True


In [2]:
from itertools import combinations
from tempfile import TemporaryDirectory

if DATA.exists():
    print("Data/ already exists: SKIPPING generation to protect the reporting set.")
else:
    raw_files = sorted(RAW.glob("subject*.dat"))
    expected_raw = [f"subject{i}.dat" for i in range(101, 110)]
    if [p.name for p in raw_files] != expected_raw:
        raise FileNotFoundError(
            "Missing original PAMAP2 Protocol files. Expected subject101.dat "
            "through subject109.dat inside PAMAP2_Dataset/Protocol/."
        )

    with TemporaryDirectory(prefix="pamap2_build_", dir=ROOT) as tmp:
        stage = Path(tmp)
        all_dir = stage / "all_subjects"
        all_dir.mkdir()
        counts = {}
        for source in raw_files:
            frame = pd.read_csv(source, sep=r"\s+", header=None)
            if frame.shape[1] != 54:
                raise ValueError(f"{source.name}: expected 54 columns, got {frame.shape[1]}")
            reduced = frame.iloc[::10].copy()
            target = all_dir / (source.stem + ".csv.gz")
            reduced.to_csv(target, index=False, header=False, compression="gzip")
            counts[source.stem] = len(reduced)
            print(f"{source.stem}: {len(reduced):,} downsampled rows")
            del frame, reduced

        total = sum(counts.values())
        candidates = []
        for pair in combinations(sorted(counts), 2):
            reporting_n = sum(counts[s] for s in pair)
            candidates.append((abs(reporting_n / total - 0.2), pair, reporting_n))
        _, pair, reporting_n = min(candidates)
        reporting_ids = sorted(pair)
        training_ids = sorted(set(counts) - set(pair))
        training_n = total - reporting_n

        actual = dict(
            training_rows=training_n,
            reporting_rows=reporting_n,
            training_participants=training_ids,
            reporting_participants=reporting_ids,
        )
        if actual != EXPECTED:
            raise ValueError(
                "The newly computed split does not match the finalized project split. "
                f"Actual: {actual}. Nothing was written to Data/."
            )

        train_dir = stage / "training_set"
        report_dir = stage / "reporting_set"
        train_dir.mkdir()
        report_dir.mkdir()
        for subject in sorted(counts):
            target_dir = report_dir if subject in pair else train_dir
            (all_dir / (subject + ".csv.gz")).rename(target_dir / (subject + ".csv.gz"))
        all_dir.rmdir()

        metadata = {
            "dataset": "PAMAP2 Protocol, downsampled to 10 Hz",
            **actual,
            "training_percentage": round(training_n / total * 100, 2),
            "reporting_percentage": round(reporting_n / total * 100, 2),
        }
        (stage / "split_info.json").write_text(json.dumps(metadata, indent=2))

        if DATA.exists():
            raise FileExistsError("Data/ appeared during preparation; refusing to overwrite.")
        DATA.mkdir()
        for item in list(stage.iterdir()):
            item.rename(DATA / item.name)
        print("Created Data/ with the official split.")


Data/ already exists: SKIPPING generation to protect the reporting set.


In [3]:
if not DATA.is_dir():
    raise FileNotFoundError("Data/ not found.")

with (DATA / "split_info.json").open() as f:
    metadata = json.load(f)

for key, expected in EXPECTED.items():
    assert metadata[key] == expected, (
        f"Unexpected {key}: {metadata[key]} (expected {expected})"
    )

def names_in(folder):
    return sorted(p.name.removesuffix(".csv.gz")
                  for p in folder.glob("subject*.csv.gz"))

assert names_in(DATA / "training_set") == EXPECTED["training_participants"]
assert names_in(DATA / "reporting_set") == EXPECTED["reporting_participants"]

total = metadata["training_rows"] + metadata["reporting_rows"]
print("Dataset is correctly organized.")
print("Total:", f"{total:,}")
print("Training:", f"{metadata['training_rows']:,}", f"({metadata['training_rows']/total:.2%})")
print("Reporting:", f"{metadata['reporting_rows']:,}", f"({metadata['reporting_rows']/total:.2%})")
print("No reporting-set measurements were opened or modified.")


Dataset is correctly organized.
Total: 287,257
Training: 229,015 (79.72%)
Reporting: 58,242 (20.28%)
No reporting-set measurements were opened or modified.


In [4]:
train = pd.concat(
    [pd.read_csv(p, header=None).assign(subject=p.stem.split('.')[0])
     for p in sorted(Path("Data/training_set").glob("*.csv.gz"))],
    ignore_index=True,
)
report = pd.concat(
    [pd.read_csv(p, header=None).assign(subject=p.stem.split('.')[0])
     for p in sorted(Path("Data/reporting_set").glob("*.csv.gz"))],
    ignore_index=True,
)


In [5]:
imu = ["temp",
       "acc16_x", "acc16_y", "acc16_z",
       "acc6_x", "acc6_y", "acc6_z",
       "gyro_x", "gyro_y", "gyro_z",
       "mag_x", "mag_y", "mag_z",
       "orient_1", "orient_2", "orient_3", "orient_4"]

cols = (["timestamp", "activity_id", "heart_rate"]
        + [f"hand_{c}" for c in imu]
        + [f"chest_{c}" for c in imu]
        + [f"ankle_{c}" for c in imu])

ACTIVITIES = {
    1: "lying", 2: "sitting", 3: "standing", 4: "walking", 5: "running",
    6: "cycling", 7: "nordic_walking", 9: "watching_tv", 10: "computer_work",
    11: "car_driving", 12: "ascending_stairs", 13: "descending_stairs",
    16: "vacuum_cleaning", 17: "ironing", 18: "folding_laundry",
    19: "house_cleaning", 20: "playing_soccer", 24: "rope_jumping",
}

def load_split(folder):
    frames = []
    for p in sorted(Path(folder).glob("subject*.csv.gz")):
        df = pd.read_csv(p, header=None, names=cols)
        df["subject"] = int(p.name[7:10])   # "subject101.csv.gz" -> 101
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

train = load_split("Data/training_set")
train = train[train["activity_id"] != 0].copy()          # drop transient rows
train["activity"] = train["activity_id"].map(ACTIVITIES)

train.shape, train["activity"].value_counts()


((153715, 56),
 activity
 ironing              18572
 walking              17781
 nordic_walking       16056
 lying                14744
 standing             14469
 cycling              14191
 vacuum_cleaning      13500
 sitting              13093
 running               9820
 ascending_stairs      9014
 descending_stairs     7539
 rope_jumping          4936
 Name: count, dtype: int64)

In [6]:
train

,timestamp,activity_id,heart_rate,hand_temp,hand_acc16_x,hand_acc16_y,hand_acc16_z,hand_acc6_x,hand_acc6_y,hand_acc6_z,...,ankle_gyro_z,ankle_mag_x,ankle_mag_y,ankle_mag_z,ankle_orient_1,ankle_orient_2,ankle_orient_3,ankle_orient_4,subject,activity
293,37.68,1,NaN,30.375,2.29090,7.14240,5.82342,2.26966,7.66268,5.78846,...,-0.004882,-60.3407,-35.784200,-58.611900,1.000000,0.000000,0.000000,0.000000,101,lying
294,37.78,1,NaN,30.375,2.28661,7.67154,5.59065,2.31412,7.72263,5.63727,...,-0.001803,-60.8735,-37.535300,-58.254900,1.000000,0.000000,0.000000,0.000000,101,lying
295,37.88,1,NaN,30.375,2.16379,8.28577,6.27951,2.18617,7.83003,6.45261,...,0.007628,-60.6611,-36.769000,-58.372300,1.000000,0.000000,0.000000,0.000000,101,lying
296,37.98,1,NaN,30.375,2.27181,8.09419,6.16631,2.24558,7.99547,6.18037,...,0.006608,-61.1142,-36.637900,-58.738500,1.000000,0.000000,0.000000,0.000000,101,lying
297,38.08,1,NaN,30.375,2.43405,6.76050,5.67290,2.68729,6.84442,6.09184,...,-0.012476,-60.7757,-36.868800,-58.867500,1.000000,0.000000,0.000000,0.000000,101,lying
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228959,94.67,24,NaN,25.125,4.89403,6.56286,6.94201,4.77218,6.66398,6.87534,...,-0.001103,-46.1741,-0.554862,-0.171827,0.522896,-0.295335,0.700024,-0.386424,109,rope_jumping
228960,94.77,24,162.0,25.125,4.34732,6.90337,6.35537,4.39582,6.96850,6.61834,...,0.009044,-45.5269,-0.843791,0.114449,0.522768,-0.293886,0.701447,-0.385119,109,rope_jumping
228961,94.87,24,NaN,25.125,4.63360,6.51829,5.97608,4.59745,6.36248,6.10604,...,-0.037867,-45.6445,-1.094160,0.539822,0.522604,-0.293896,0.703075,-0.382357,109,rope_jumping
228962,94.97,24,NaN,25.125,4.88770,6.55150,5.67187,4.88192,6.52618,5.71277,...,-0.017305,-46.0300,-1.334520,-0.170416,0.523125,-0.292440,0.704461,-0.380202,109,rope_jumping


In [7]:
report = load_split("Data/reporting_set")
report = report[report["activity_id"] != 0].copy()        # drop transient rows
report["activity"] = report["activity_id"].map(ACTIVITIES)

report.shape, report["activity"].value_counts()


((40577, 56),
 activity
 walking              6097
 sitting              5425
 ironing              5296
 standing             4525
 lying                4510
 vacuum_cleaning      4037
 descending_stairs    2955
 nordic_walking       2754
 ascending_stairs     2708
 cycling              2270
 Name: count, dtype: int64)

In [8]:
report

,timestamp,activity_id,heart_rate,hand_temp,hand_acc16_x,hand_acc16_y,hand_acc16_z,hand_acc6_x,hand_acc6_y,hand_acc6_z,...,ankle_gyro_z,ankle_mag_x,ankle_mag_y,ankle_mag_z,ankle_orient_1,ankle_orient_2,ankle_orient_3,ankle_orient_4,subject,activity
1605,166.14,1,NaN,31.0000,-1.843770,4.50090,8.04115,-1.625620,4.58187,8.09451,...,0.114357,-42.9115,0.921397,56.87510,0.691223,-0.018539,0.722390,-0.004570,103,lying
1606,166.24,1,NaN,31.0000,-2.972890,8.31757,9.55086,-3.062250,9.41154,9.86729,...,-0.297600,-41.7412,-1.786710,58.32190,0.683921,0.001872,0.729346,0.017415,103,lying
1607,166.34,1,NaN,31.0000,-1.502600,4.46067,8.08489,-1.519500,4.50574,8.24557,...,-0.709892,-38.2063,-5.025710,60.23890,0.662897,-0.010936,0.747210,0.046095,103,lying
1608,166.44,1,NaN,31.0000,-0.368378,4.52779,8.10165,-0.416785,5.02615,8.01679,...,-0.399414,-34.2511,-6.616690,61.55120,0.641754,-0.023770,0.763274,0.070706,103,lying
1609,166.54,1,NaN,31.0000,-0.967945,5.25331,8.09022,-1.150430,5.46953,8.15248,...,0.620695,-32.1073,-7.037620,62.64320,0.632683,-0.016189,0.770804,0.072880,103,lying
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57439,3221.21,6,NaN,27.5625,-8.599050,4.34528,1.70553,-8.734130,4.38557,1.68438,...,-0.090457,-41.3720,0.650334,9.48825,0.721457,-0.120620,0.677857,-0.073901,104,cycling
57440,3221.31,6,NaN,27.5625,-8.787990,4.15738,1.74183,-8.825960,4.09940,1.80591,...,0.002904,-41.2333,-0.389470,8.49712,0.723263,-0.112038,0.678178,-0.066429,104,cycling
57441,3221.41,6,131.0,27.5625,-8.785320,4.15805,1.81884,-8.826210,4.11446,1.76057,...,0.024773,-40.9744,-0.271965,9.06522,0.723968,-0.103498,0.679163,-0.062423,104,cycling
57442,3221.51,6,NaN,27.5625,-8.673280,4.15684,1.78203,-8.749470,4.21968,1.82067,...,-0.016745,-40.8529,-0.279699,8.49833,0.725473,-0.095206,0.679265,-0.056786,104,cycling


In [9]:
# Fill missing sensor values within each subject (heart_rate is ~91% NaN due to lower sampling rate)
for df in (train, report):
    num_cols = df.columns.drop(["subject", "activity", "activity_id"])
    df[num_cols] = df.groupby("subject")[num_cols].transform(lambda s: s.ffill().bfill())

train.isna().sum().sum(), report.isna().sum().sum()

(np.int64(0), np.int64(0))

In [10]:
drop_cols = ["activity_id", "activity", "subject", "timestamp"] + \
            [c for c in train.columns if "orient" in c]

X_train = train.drop(columns=drop_cols)
y_train = train["activity"]
### Groupmate: the groups are for cross-validation so that the cv can match the subjects to the data.
groups_train = train["subject"]

X_report = report.drop(columns=drop_cols)
y_report = report["activity"]

assert list(X_train.columns) == list(X_report.columns)
assert not X_train.isna().any().any() and not X_report.isna().any().any()
X_train.shape, X_report.shape

((153715, 40), (40577, 40))